<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026psy3a_lect05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib

## Flanker課題の計算モデル（S-S競合のシミュレーション）

このモデルでは、情報を「入力層」→「表象層（意味）」→「出力層（運動）」へと伝播させます。
Flanker 課題のポイントは、**ターゲット刺激と妨害刺激が、同じ「表象層」に流れ込んで競合を起こす（S-S競合）** という点です。

* **Congruent（一致条件）**: 中央「左」＋ 周辺「左」
* **Incongruent（不一致条件）**: 中央「左」＋ 周辺「右」

下のコードを実行し、横軸（時間ステップ）に対する縦軸（正答の出力ノードの活性化量）の立ち上がり方の違いを観察してください。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_flanker(condition, cycles=80):
    # 1. パラメータ設定
    tau = 0.1             # 情報の伝播速度（カスケード率）
    inhibition = -0.3     # 側抑制の強さ
    weight_center = 1.0   # 中央（ターゲット）への注意の重み
    weight_flanker = 0.4  # 周辺（妨害刺激）への注意の漏れ

    # 側抑制の重み行列（対角成分は0、非対角成分がお互いを抑制する）
    W_inh = np.array([[0.0, inhibition],
                      [inhibition, 0.0]])

    # 各層の初期化（インデックス 0:「左」, 1:「右」）
    hidden_layer = np.zeros(2) # 表象層
    output_layer = np.zeros(2) # 出力層

    # 2. 入力刺激の定義
    # ターゲットは常に「左」を向いていると仮定
    input_center = np.array([1.0, 0.0])

    if condition == "congruent":
        input_flanker = np.array([1.0, 0.0]) # 周辺も「左」
    elif condition == "incongruent":
        input_flanker = np.array([0.0, 1.0]) # 周辺は「右」
    else:
        raise ValueError("無効な条件が指定されました")

    history_correct = []

    # 3. シミュレーションループ
    for _ in range(cycles):

        # --- 表象層（S-S競合の発生場所）の更新 ---
        # 外部からの入力
        ext_input_hidden = (input_center * weight_center) + (input_flanker * weight_flanker)
        # 表象層内での側抑制（行列の内積による一括計算）
        inh_input_hidden = np.dot(hidden_layer, W_inh)

        # 総入力と活性化値の更新（ReLU 的処理: 0未満にはならない）
        net_hidden = ext_input_hidden + inh_input_hidden
        hidden_layer = np.maximum(0, hidden_layer + tau * net_hidden)

        # --- 出力層の更新 ---
        # 表象層からの入力
        ext_input_output = hidden_layer.copy()
        # 出力層内での側抑制（行列の内積による一括計算）
        inh_input_output = np.dot(output_layer, W_inh)

        # 総入力と活性化値の更新
        net_output = ext_input_output + inh_input_output
        output_layer = np.maximum(0, output_layer + tau * net_output)

        # 正答（左）ノードの活性化量を記録
        history_correct.append(output_layer)

    return history_correct

# --- 実行と結果の可視化 ---
cycles = 80
history_cong = simulate_flanker("congruent", cycles)
history_incong = simulate_flanker("incongruent", cycles)

plt.figure(figsize=(10, 6))
plt.plot(history_cong, label="一致条件 (中央「左」-周辺「左」)", color="blue", linewidth=3)
plt.plot(history_incong, label="不一致条件 (中央「左」-周辺「右」)", color="red", linewidth=3)

# 意思決定の閾値
plt.axhline(y=2.0, color='gray', linestyle='--', label="意思決定しきい値")

plt.title("Flanker 課題シミュレーション (S-S 葛藤)", fontsize=14)
plt.xlabel("時間ステップ (処理時間)", fontsize=12)
plt.ylabel("正解出力ノードの活性値", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()


# Simon 課題

Flanker課題との決定的な違いは、空間位置情報の入力が「表象層」を完全にバイパスし、直接「出力層」へ流れ込む結線（自動的経路）が存在する点にある。コード内の ext_input_output の算出ロジックに注意

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_simon(condition, cycles=100):
    # 1. パラメータ設定
    tau = 0.1             # 情報の伝播速度（カスケード率）
    inhibition = -0.3     # 側抑制の強さ

    # 経路の重み（Simon 課題のコアアーキテクチャ）
    weight_color_to_rep = 1.0  # 制御的経路（色情報 → 表象層）
    weight_rep_to_out = 1.0    # 制御的経路（表象層 → 出力層）
    weight_loc_to_out = 0.3    # 自動的経路（位置情報 → 出力層へのダイレクトバイパス）

    # 側抑制の重み行列
    W_inh = np.array([[0.0, inhibition],
                      [inhibition, 0.0]])

    # 各層の初期化（インデックス 0:「左」, 1:「右」）
    hidden_layer = np.zeros(2) # 表象層
    output_layer = np.zeros(2) # 出力層

    # 2. 入力刺激の定義
    # ターゲット色（例：赤なら「左」ボタンを押すというルール）
    input_color = np.array([1.0, 0.0])

    # 刺激の空間的な提示位置
    if condition == "congruent":
        input_location = np.array([1.0, 0.0]) # 左側に提示
    elif condition == "incongruent":
        input_location = np.array([0.0, 1.0]) # 右側に提示
    else:
        raise ValueError("無効な条件が指定されました")

    history_correct = []
    history_incorrect = [] # Simon 課題の特異性を観察するため、誤答ノードも記録

    # 3. シミュレーションループ
    for _ in range(cycles):

        # --- 表象層の更新 ---
        # 注意: Simon課題では、空間位置からの入力はここには入らない
        ext_input_hidden = input_color * weight_color_to_rep
        inh_input_hidden = np.dot(hidden_layer, W_inh)

        net_hidden = ext_input_hidden + inh_input_hidden
        hidden_layer = np.maximum(0, hidden_layer + tau * net_hidden)

        # --- 出力層（S-R競合の発生場所）の更新 ---
        # 制御的経路（表象層から） ＋ 自動的経路（空間位置からのダイレクトバイパス）
        ext_input_output = (hidden_layer * weight_rep_to_out) + (input_location * weight_loc_to_out)
        inh_input_output = np.dot(output_layer, W_inh)

        net_output = ext_input_output + inh_input_output
        output_layer = np.maximum(0, output_layer + tau * net_output)

        # 活性化量を記録（0: 正解ノード, 1: 誤答ノード）
        history_correct.append(output_layer)
        history_incorrect.append(output_layer[1])

    return history_correct, history_incorrect

# --- 実行と結果の可視化 ---
cycles = 100
history_cong_corr, _ = simulate_simon("congruent", cycles)
history_incong_corr, history_incong_incorr = simulate_simon("incongruent", cycles)

plt.figure(figsize=(10, 6))
# 正答ノードの活性化推移
plt.plot(history_cong_corr, label="一致条件 - 正解ノード", color="blue", linewidth=3)
plt.plot(history_incong_corr, label="不一致条件 - 正解ノード", color="red", linewidth=3)

# 誤答ノードの活性化推移（S-R競合の証拠）
plt.plot(history_incong_incorr, label="不一致条件 - 不正解ノード (干渉)", color="orange", linestyle='--', linewidth=2)

# 意思決定の閾値
plt.axhline(y=2.0, color='gray', linestyle='--', label="意思決定しきい値")

plt.title("Simon 課題シミュレーション (S-R コンフリクト)", fontsize=14)
plt.xlabel("時間ステップ (処理時間)", fontsize=12)
plt.ylabel("活性値", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

教授法に対する戦略的要件
このコードを提示し、グラフを描画させたのち、学生に対して以下の分析を強制せよ。

* オレンジ色の破線の意味は何か？

不一致条件（Incongruent）において、オレンジ色の破線（誤答ノード）が序盤で一時的に活性化し、その後減衰している。
なぜ正解でもないノードが勝手に立ち上がるのか？
それは `weight_loc_to_out` による自動的経路が、遅れてやってくる `hidden_layer`（色情報の処理）よりも先に運動出力を駆動しようとするからである。
このグラフの挙動こそが、S-R 競合の数理的証明である。
現象を暗記させるのではなく、このグラフからアーキテクチャの構造を逆算させよ。

* Flankerモデルとの構造的差異は

Flanker モデルのコードと Simon モデルのコードを並べ、`ext_input_hidden` と `ext_input_output` の記述がどのように異なるか比較せよ。
Flanker では表象層でノイズが混入したが、Simon では出力層でノイズが混入する。
この「変数の足し算が行われる場所の違い」が、心理学における「情報処理段階の違い」と完全に同値である

# Stroop 課題

Stroop 課題と Flanker 課題は、ネットワーク上の構造（競合の発生場所）は完全に同一である」。
両者とも「入力層」から「表象層」へのマッピングにおいて発生する刺激競合（S-S 競合）にすぎない。

では、なぜ Stroop 効果は Flanker 効果よりもあれほど強烈な干渉（遅延）を引き起こすのか？

それは「文字を読む」という行為の自動性の高さ（経路の重みの非対称性）による。Flanker 課題における周辺視野からの「注意の漏れ」が小さなノイズであるのに対し、Stroop 課題における「文字情報」の混入は、意図的な制御（色を答える）を凌駕しかねないほどの強烈な信号として表象層を直撃する。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_stroop(condition, cycles=120):
    # 1. パラメータ設定
    tau = 0.1             # 情報の伝播速度
    inhibition = -0.3     # 側抑制の強さ

    # Stroop課題の核心：経路の非対称性（自動性）
    # 色呼称は課題要求（トップダウン注意）によって重みが高められているが...
    weight_color = 1.5
    # 文字読みは極めて自動性が高く、注意を向けずとも強烈な信号を送る
    # （Flanker課題の weight_flanker = 0.4 と比較せよ）
    weight_word = 1.2

    # 側抑制の重み行列
    W_inh = np.array([[0.0, inhibition],
                      [inhibition, 0.0]])

    # 各層の初期化（インデックス 0:「赤」, 1:「緑」）
    hidden_layer = np.zeros(2) # 意味表象層
    output_layer = np.zeros(2) # 出力層

    # 2. 入力刺激の定義
    # ターゲット（インクの色）：常に「赤（インデックス0）」とする
    input_color = np.array([1.0, 0.0])

    # 妨害刺激（書かれている文字）
    if condition == "congruent":
        input_word = np.array([1.0, 0.0]) # 文字も「あか」
    elif condition == "incongruent":
        input_word = np.array([0.0, 1.0]) # 文字は「みどり」
    else:
        raise ValueError("無効な条件が指定されました")

    history_correct = []
    history_incorrect = []

    # 3. シミュレーションループ
    for _ in range(cycles):

        # --- 表象層（S-S競合の発生場所）の更新 ---
        # 構造はFlankerと全く同じ。しかし weight_word が大きいため強烈な競合が起きる
        ext_input_hidden = (input_color * weight_color) + (input_word * weight_word)
        inh_input_hidden = np.dot(hidden_layer, W_inh)

        net_hidden = ext_input_hidden + inh_input_hidden
        hidden_layer = np.maximum(0, hidden_layer + tau * net_hidden)

        # --- 出力層の更新 ---
        ext_input_output = hidden_layer.copy()
        inh_input_output = np.dot(output_layer, W_inh)

        net_output = ext_input_output + inh_input_output
        output_layer = np.maximum(0, output_layer + tau * net_output)

        history_correct.append(output_layer)
        history_incorrect.append(output_layer[1])

    return history_correct, history_incorrect

# --- 実行と結果の可視化 ---
cycles = 120
history_cong_corr, _ = simulate_stroop("congruent", cycles)
history_incong_corr, history_incong_incorr = simulate_stroop("incongruent", cycles)

plt.figure(figsize=(10, 6))
# 正答ノードの活性化推移
plt.plot(history_cong_corr, label="一致条件 (インクの色:赤, 単語:赤)", color="blue", linewidth=3)
plt.plot(history_incong_corr, label="不一致条件 (インクの色:赤, 単語:緑)", color="red", linewidth=3)

# 誤答ノードの活性化推移
plt.plot(history_incong_incorr, label="Incongruent - Incorrect Node (Word Interference)", color="orange", linestyle='--', linewidth=2)

# 意思決定の閾値
plt.axhline(y=3.0, color='gray', linestyle='--', label="意思決定のしきい値")

plt.title("Stroop 課題シミュレーション (自動 S-S コンフリクト)", fontsize=14)
plt.xlabel("時間ステップ (処理時間)", fontsize=12)
plt.ylabel("活性値", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()